# Board Game Video Detection Project
Jakub Laskowski 160287
Jakub Górniak 160326

## 1. Board Game - Man, Don't Get Angry (w/ coins)

### 1.1 Description
It is a board game for 2-4 players. Each player has 4 pawns of their color that start in their home base. Players take turns rolling a dice to move their pawns around the board, trying to get all four pawns from the start area to their finish zone. A roll of 6 is required to move a pawn onto the board and grants an extra turn. If a pawn lands on a space occupied by an opponent's pawn, the opponent's pawn is captured and sent back to their home base. In our extended version, capturing an opponent's pawn awards the capturing player a coin token.

![title](board.jpg)


### 1.2 Items
- Game board - Cross-shaped board with 4 colored corners (home bases), 4 finish zones
- Pawns – 16 total (4 per player in 4 different colors)
- Die – Standard 6 sided die
- Coin tokens – Awarded when a player captures an opponent's pawn


### 1.3 Events
- Dice roll - Reading the value on the dice
- Pawn entering the board - and bonus turn (rolling 6)
- Moving a pawn
- Capture oponent/Coin collection - A pawn lands on an opponent's pawn, sending it back to home base, capturing player recieves a coin token
- Pawn entering finish zone/No more moves - A pawn moves into the finish zone and can no longer be moved
- Game start – All pawns are positioned in their respective home bases
- Player wins/Game end – A player successfully moves all 4 of their pawns into the finish zone


## 2. Dataset


### 2.1 Easy
Videos were recorded from a top-down view with lighting conditions as good as possible. Every part of the game was clearly visible. Players hand obstructed the pieces only when it was needed (due to the size and shape of the board many times it was impossible to not cover part of the game).
![easy](first_frame_easy_1.jpg)

### 2.2 Medium
Same as for easy dataset, except the lighting conditions were not as good and created shadows, light reflections, darker board.
![medium](first_frame_medium_2.jpg)

### 2.3 Difficult
Same as for medium, except the angle of the camera had several degrees with slight shaking sometimes and the hand obstructed the game pieces more often.
![diffcult](first_frame_hard_3.jpg) 


## 3. Methods used


### 3.1 Dice detection
The detection function isolates the dice by converting the input frame to the HSV color space and generating a binary mask based on brightness and saturation thresholds. It refines this mask using morphological closing and opening operations, which fill internal holes and remove background noise to ensure a solid shape. The function then filters potential candidates by analyzing geometric properties, rejecting contours that do not meet specific area and circularity criteria typical of a die. Finally, it determines the dice value using adaptive thresholding, which dynamically separates black pips from the white surface by calculating local thresholds, ensuring accuracy despite uneven lighting. There are multiple methods used to detect the pips and they vote on what the value should be.

![dice_1](dice_det_1.png)
![dice_2](dice_det_2.png)


### 3.2 Board detection
To isolate the board from the background, we began by converting the input frame to grayscale and applying a Gaussian blur to reduce image noise. Later rather than relying on a single binarization parameter, we generated distinct binary candidate maps utilizing three diverse techniques: Otsu’s method for global threshold optimization, a fixed empirical threshold, and adaptive Gaussian thresholding. Each candidate map underwent morphological opening and closing. We then performed contour analysis for all candidates, evaluating the geometric properties of the detected shapes to identify the best quadrilateral that maximized the board area. The vertices of this optimal candidate were subsequently mapped to a rectified coordinate system to compute the grid matrix for the perspective warp using the peaks to find the lines.

![board_1](board_det_1.png)
![board_2](board_det_2.png)

### 3.3 Pawns detection
To detect and track the game pieces, we used the grid to extract a region of interest (ROI) for each logical tile (to find each colors home, starting and finish tiles). Pixel analysis was conducted using the HSV color space. We implemented a heuristic evaluation algorithm that applied specific hue thresholds for the four player teams. For each ROI, a confidence metric was computed by analyzing the fill ratio—specifically contrasting the pixel density of the target color in the tile's center against its periphery. To ensure the stability of the system and possible missdetection due to obstructing the pieces, these detections were processed through a temporal tracking filter. This maintained a history buffer for every candidate, validating a piece's presence only after it was consistently observed across a predefined sequence of frames, thereby suppressing flickering.

![pawns](pawn_det.png)

### 3.4 Token detection
The detection function isolates tokens by converting the input frame to the HSV color space and generating a binary mask from two separate hue ranges to capture the red color spectrum's wrap-around nature. It improves this mask using morphological closing and opening operations, to eliminate background noise and produce clean, continuous contours. Then it filters these contours by analyzing their geometric properties, rejecting candidates that do not meet area and circularity thresholds of those tokens. Finally, it employs Euclidean distance tracking to associate new detections with existing token identities, ensuring consistent tracking across consecutive frames despite object movement.

![token_1](token_det_1.png)
![token_2](token_det_2.png)

## 4. Effectiveness
In the end we implemented detection of 5 events for each level of dataset difficulty:
- Dice roll - Reading the value on the dice
- Pawn entering the board - and bonus turn (rolling 6)
- Capture opponent/Coin collection - A pawn lands on an opponent's pawn, sending it back to home base, capturing player recieves a coin token
- Game start – All pawns are positioned in their respective home bases
- Player wins/Game end – A player successfully moves all 4 of their pawns into the finish zone

In addition, we track the:
- Where the players' pawns are currently placed on board (Home/Track/Finish)
- Tracking the dice/Rolling

### 4.1 Easy
Our solution was very effective on the easy difficulty. There were no issues with tracking, only when a players hand covered the board the pieces would disappear for a while but without any false detections.

### 4.2 Medium
The solution performed similarly for this difficulty.The lighting conditions didn't cause big issues for detecting the board and its pieces. The overall accuracy was almost the same as for easy dataset. 

### 4.3 Difficult
As for the highest difficulty the solution also performed really well in detecting pawns, board and tokens. However the hand movements covering the pieces and the angle, sometimes cause the pieces to be detected in the adjacent tiles or disappear more often than in lower difficulties. The biggest issue we saw with the dice value detection as the angle showed the side of the dice which casued some inaccuracies in the value detection.

## 5. Analysis and Conclusion
Throughout this project, we developed a computer vision pipeline designed to detect dice values and track game tokens on a Man Don't Get Angry board. While the system demonstrated great performance in identifying tokens and reading dice pips under controlled conditions, the development process highlighted critical challenges related to object segmentation.

A primary technical challenge was the accurate detection of the board grid. Initial attempts using the standard Hough Line Transform proved unreliable, the algorithm struggled to consistently identify grid lines amidst the visual noise of the pawns and varying lighting conditions, often resulting in fragmented, missing segments or non-existing lines. To overcome this, we shifted to a "projection peak" method. By summing pixel intensities along the x and y axes, we identified clear peaks corresponding to grid lines. This approach proved far more stable and deterministic than Hough lines, allowing us to reconstruct the grid structure reliably even when individual line segments were faint.

Similarly, tracking player pawns presented significant difficulties. Our initial approach relied solely on global color intensity thresholding, which proved highly vulnerable to environmental changes like shadows and reflections, often resulting in false positives or lost tracking. We addressed this by shifting to a spatially-aware, grid-based scanning pipeline. Instead of scanning the entire image at once, the system isolates each board tile and evaluates it using a local density scoring metric. This metric prioritizes color saturation in the center of a tile while penalizing spillover at the borders, effectively filtering out noise. To handle angled views, we developed a "ghost removal" heuristic that analyzes spatial relationships along grid lanes to suppress duplicate detections caused by perspective overlap. Finally, object permanence was enforced through a temporal history buffer with specific "lock" thresholds, ensuring consistent piece identities even during momentary occlusions or lighting flickers.

Despite these difficulties, the final implementation achieved significant accuracy through all recording. The integration of peak-based grid detection ensured the game board was correctly mapped, while the enhanced tracking logic allowed for the reliable monitoring of game pieces, the project demonstrated the potential of combining computer vision methods to analyze complex, dynamic recordings effectively.